In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


#STAGE 1 : IMPORTS

In [9]:
import pandas as pd

lap_path = '/content/drive/MyDrive/F1BigData/cleaned_batches/final_clean_laps.csv'
telemetry_path = '/content/drive/MyDrive/F1BigData/raw/2023_SpanishGP/telemetry.csv'

laps_df = pd.read_csv(lap_path)
tele_df = pd.read_csv(telemetry_path)

print("Lap data shape:", laps_df.shape)
print("Telemetry data shape:", tele_df.shape)
print("Lap columns:", laps_df.columns.tolist())
print("Telemetry columns:", tele_df.columns.tolist())

Lap data shape: (1312, 31)
Telemetry data shape: (20014, 11)
Lap columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate']
Telemetry columns: ['Date', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Time', 'SessionTime', 'Distance']


In [10]:
print(laps_df.head(3))
print(tele_df.head(3))

                     Time Driver  DriverNumber  LapTime  LapNumber  Stint  \
0  0 days 01:03:40.107000    VER             1      NaN        1.0    1.0   
1  0 days 01:05:00.509000    VER             1      NaN        2.0    1.0   
2  0 days 01:06:21.008000    VER             1      NaN        3.0    1.0   

  PitOutTime PitInTime             Sector1Time             Sector2Time  ...  \
0    Unknown   Unknown                 Unknown  0 days 00:00:32.084000  ...   
1    Unknown   Unknown  0 days 00:00:24.186000  0 days 00:00:32.088000  ...   
2    Unknown   Unknown  0 days 00:00:24.167000  0 days 00:00:32.191000  ...   

  FreshTyre             Team            LapStartTime             LapStartDate  \
0      True  Red Bull Racing  0 days 01:02:15.963000  2023-06-04 13:03:15.970   
1      True  Red Bull Racing  0 days 01:03:40.107000  2023-06-04 13:04:40.114   
2      True  Red Bull Racing  0 days 01:05:00.509000  2023-06-04 13:06:00.516   

   TrackStatus  Position  Deleted  DeletedReason 

#STAGE 2 : CLEAN TELEMETRY DATA, TESTING

In [17]:
import os
import numpy as np
from datetime import datetime

tele_clean = tele_df.copy()

# convert columns
tele_clean['Date'] = pd.to_datetime(tele_clean['Date'], errors='coerce')
for col in ['Time', 'SessionTime', 'Distance']:
    tele_clean[col] = pd.to_numeric(tele_clean[col], errors='coerce')

tele_clean['nGear'] = tele_clean['nGear'].fillna(0).astype(int)

# fill missing numeric values
for col in ['Throttle', 'Brake', 'DRS', 'RPM', 'Speed']:
    if col in tele_clean.columns:
        tele_clean[col] = tele_clean[col].interpolate(limit_direction='both').fillna(0)

# remove duplicates & unrealistic values
tele_clean = tele_clean.drop_duplicates()
tele_clean = tele_clean[
    (tele_clean['Speed'] >= 0) &
    (tele_clean['Throttle'].between(0, 100)) &
    (tele_clean['Brake'].between(0, 1)) &
    (tele_clean['RPM'] >= 0)
].reset_index(drop=True)

print("Telemetry cleaned:", tele_clean.shape)

Telemetry cleaned: (20014, 11)


##Tetsing, cleaning, validation

In [27]:
base_dir = '/content/drive/MyDrive/F1BigData'
clean_dir = os.path.join(base_dir, 'cleaned_batches')
log_dir = os.path.join(base_dir, 'logs')
os.makedirs(clean_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

lap_clean = pd.read_csv(os.path.join(clean_dir,'final_clean_laps.csv'))
tele_clean = pd.read_csv(os.path.join(base_dir, 'raw', '2023_SpanishGP', 'telemetry.csv'))

def validate_telemetry(df):
    errors = []
    numeric_cols = ['RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Time', 'SessionTime', 'Distance']
    for col in numeric_cols:
        if not np.issubdtype(df[col].dtype, np.number):
            errors.append(f"{col}: Non-numeric type ({df[col].dtype})")
        elif df[col].isna().any():
            errors.append(f"{col}: {df[col].isna().sum()} missing values")

    if not df['Throttle'].between(0, 100).all():
        errors.append("Throttle values out of range [0,100]")
    if not df['Brake'].between(0, 1).all():
        errors.append("Brake values out of range [0,1]")
    if (df['Speed'] < 0).any():
        errors.append("Negative speed values found")
    if (df['RPM'] < 0).any():
        errors.append("Negative RPM values found")

    if errors:
        status = "FAILED"
        print(f"{len(errors)} validation issues found.")
    else:
        status = "PASSED"
        print("Telemetry data passed all validation checks!")
    return status, errors

tele_clean = tele_clean.copy()

# convert Brake (bool → int)
if tele_clean['Brake'].dtype == bool:
    tele_clean['Brake'] = tele_clean['Brake'].astype(int)

# handle missing Time/SessionTime
if tele_clean['Time'].isna().all() and tele_clean['SessionTime'].isna().all():
    print("Missing time columns — reconstructing from distance.")
    avg_speed = tele_clean['Speed'].replace(0, np.nan).mean()
    tele_clean['Time'] = np.cumsum(tele_clean['Distance'].diff().fillna(0) / avg_speed)
    tele_clean['SessionTime'] = tele_clean['Time']
else:
    tele_clean['Time'] = tele_clean['Time'].fillna(method='ffill').fillna(0)
    tele_clean['SessionTime'] = tele_clean['SessionTime'].fillna(method='ffill').fillna(0)

for col in ['Time', 'SessionTime']:
    if col in tele_clean.columns:
        tele_clean[col] = pd.to_numeric(tele_clean[col], errors='coerce')

# fill any remaining NaN with 0 or ffill
tele_clean['Time'] = tele_clean['Time'].fillna(method='ffill').fillna(0)
tele_clean['SessionTime'] = tele_clean['SessionTime'].fillna(method='ffill').fillna(0)

print("\n Running telemetry validation after fixes...") #validate
status, issues = validate_telemetry(tele_clean)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

tele_clean_path = os.path.join(clean_dir, f"telemetry_clean_{timestamp}.csv")
tele_clean.to_csv(tele_clean_path, index=False)

report_path = os.path.join(log_dir, f"telemetry_validation_{timestamp}.txt")
with open(report_path, "w") as f:
    f.write("Telemetry Data Validation Report\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Status: {status}\n\n")
    if issues:
        f.write("Issues Found:\n")
        for issue in issues:
            f.write(f"- {issue}\n")
    else:
        f.write("No issues found. All checks passed!\n")

print(f"Cleaned telemetry saved to: {tele_clean_path}")
print(f"Validation report saved to: {report_path}")

/tmp/ipython-input-1816000282.py:49: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tele_clean['Time'] = tele_clean['Time'].fillna(method='ffill').fillna(0)
/tmp/ipython-input-1816000282.py:50: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tele_clean['SessionTime'] = tele_clean['SessionTime'].fillna(method='ffill').fillna(0)
/tmp/ipython-input-1816000282.py:57: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tele_clean['Time'] = tele_clean['Time'].fillna(method='ffill').fillna(0)
/tmp/ipython-input-1816000282.py:58: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tele_clean['SessionTime'] = tele_clean['SessionTime'].fillna(method='ffill').fi


 Running telemetry validation after fixes...
Telemetry data passed all validation checks!
Cleaned telemetry saved to: /content/drive/MyDrive/F1BigData/cleaned_batches/telemetry_clean_2025-10-29_05-41-28.csv
Validation report saved to: /content/drive/MyDrive/F1BigData/logs/telemetry_validation_2025-10-29_05-41-28.txt


##Integration - telemetry and lap data

In [35]:
print("Telemetry SessionTime range:", tele_clean['SessionTime'].min(), "→", tele_clean['SessionTime'].max())

lap_df = pd.read_csv('/content/drive/MyDrive/F1BigData/cleaned_batches/final_clean_laps.csv')
for col in ['Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime']:
    if col in lap_df.columns:
        print(f"{col} range:", lap_df[col].min(), "→", lap_df[col].max())

Telemetry SessionTime range: nan → nan
Sector1SessionTime range: 0 days 01:04:04.338000 → Unknown
Sector2SessionTime range: 0 days 01:03:16.243000 → 0 days 02:31:04.205000
Sector3SessionTime range: 0 days 01:03:40.243000 → 0 days 02:31:28.555000


In [43]:
tele_raw_path = '/content/drive/MyDrive/F1BigData/raw/2023_SpanishGP/telemetry.csv'
lap_clean_path = '/content/drive/MyDrive/F1BigData/cleaned_batches/final_clean_laps.csv'

tele = pd.read_csv(tele_raw_path)
lap = pd.read_csv(lap_clean_path)

print("Raw telemetry:", tele.shape)
print("Lap data:", lap.shape)

tele['Brake'] = tele['Brake'].astype(float)
tele['SessionTime'] = pd.to_timedelta(tele['SessionTime'], errors='coerce').dt.total_seconds()
tele['Time'] = pd.to_timedelta(tele['Time'], errors='coerce').dt.total_seconds()

tele = tele.dropna(subset=['SessionTime'])
tele = tele[tele['SessionTime'] > 0]

for col in ['Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime']:
    if col in lap.columns:
        lap[col] = pd.to_timedelta(lap[col], errors='coerce').dt.total_seconds()

lap = lap.dropna(subset=['Sector1SessionTime'])

bad_cols = [c for c in ['RPM', 'Speed', 'Throttle', 'Brake', 'SessionTime', 'Distance'] if not np.issubdtype(tele[c].dtype, np.number)]
if bad_cols:
    print("Non-numeric telemetry columns detected:", bad_cols)
else:
    print("All telemetry numeric columns are valid.")

# merge telemetry with lap data using nearest SessionTime
merged_df = pd.merge_asof(
    tele.sort_values('SessionTime'),
    lap.sort_values('Sector1SessionTime'),
    left_on='SessionTime',
    right_on='Sector1SessionTime',
    direction='nearest',
    tolerance=0.5 )  # seconds

merged_path = '/content/drive/MyDrive/F1BigData/cleaned_batches/integrated_lap_telemetry.csv'
merged_df.to_csv(merged_path, index=False)

print(f"\nIntegration successful! Merged data saved to:\n{merged_path}")
print(f"Merged shape: {merged_df.shape}")
#not accurate: lap data is NaN

Raw telemetry: (20014, 11)
Lap data: (1312, 31)
All telemetry numeric columns are valid.

Integration successful! Merged data saved to:
/content/drive/MyDrive/F1BigData/cleaned_batches/integrated_lap_telemetry.csv
Merged shape: (20014, 42)


In [44]:
print("Telemetry SessionTime range:", tele['SessionTime'].min(), "→", tele['SessionTime'].max())
sector_cols = ['Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime']
for c in sector_cols:
    if c in lap.columns:
        print(f"{c} range:", lap[c].min(), "→", lap[c].max())

Telemetry SessionTime range: 3736.143 → 9013.959
Sector1SessionTime range: 3844.338 → 9032.362
Sector2SessionTime range: 3876.426 → 9064.205
Sector3SessionTime range: 3900.554 → 9088.555


In [51]:
tele_aligned = tele.copy()

# shift SessionTime by +108 seconds to align with lap start
tele_aligned['SessionTime'] = tele_aligned['SessionTime'] + 108

merged_df = pd.merge_asof(
    tele_aligned.sort_values('SessionTime'),
    lap.sort_values('Sector1SessionTime'),
    left_on='SessionTime',
    right_on='Sector1SessionTime',
    direction='nearest',
    tolerance=2.0)

print(f"Merged shape: {merged_df.shape}")
print(merged_df.head(2))

Merged shape: (20014, 43)
                      Date      RPM  Speed  nGear  Throttle  Brake  DRS  \
0  2023-06-04 13:03:16.150  10072.0    0.0      1      16.0    1.0    1   
1  2023-06-04 13:03:16.550   8862.0    5.0      1      17.0    0.0    1   

  Source  Time_x  SessionTime  ...  FreshTyre             Team  \
0    car    0.18     3844.143  ...       True  Red Bull Racing   
1    car    0.58     3844.543  ...       True  Red Bull Racing   

             LapStartTime             LapStartDate  TrackStatus  Position  \
0  0 days 01:03:40.107000  2023-06-04 13:04:40.114          1.0       1.0   
1  0 days 01:03:40.107000  2023-06-04 13:04:40.114          1.0       1.0   

   Deleted  DeletedReason FastF1Generated IsAccurate  
0    False        Unknown           False       True  
1    False        Unknown           False       True  

[2 rows x 43 columns]


In [52]:
save_dir = '/content/drive/MyDrive/F1BigData/cleaned_batches'
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
merged_path = os.path.join(save_dir, f"merged_lap_telemetry_{timestamp}.csv")
merged_df.to_csv(merged_path, index=False)
print(f"Merged dataset saved successfully:\n{merged_path}")
print(f"Shape: {merged_df.shape}")

Merged dataset saved successfully:
/content/drive/MyDrive/F1BigData/cleaned_batches/merged_lap_telemetry_2025-10-29_06-11-42.csv
Shape: (20014, 43)


##Validation

In [53]:
def validate_merged(df):
    issues = []

    if df.empty:
        issues.append("Merged DataFrame is empty.")

    for col in ['Speed', 'RPM', 'Throttle', 'Brake']:
        if col in df.columns:
            if not np.issubdtype(df[col].dtype, np.number):
                issues.append(f"{col}: Non-numeric type ({df[col].dtype})")
            elif df[col].isna().mean() > 0.3:
                issues.append(f"{col}: {df[col].isna().mean()*100:.1f}% missing values")

    if 'LapTime' in df.columns:
        if (df['LapTime'] <= 0).any():
            issues.append("LapTime contains non-positive values.")
        if df['LapTime'].isna().all():
            issues.append("All LapTime values are missing.")

    if df.duplicated().any():
        issues.append("Duplicate rows found.")

    status = "FAILED" if issues else "PASSED"
    return status, issues

status, issues = validate_merged(merged_df)

log_dir = '/content/drive/MyDrive/F1BigData/logs'
os.makedirs(log_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
report_path = os.path.join(log_dir, f"merged_validation_{timestamp}.txt")

with open(report_path, "w") as f:
    f.write("Merged data validation report\n")
    f.write(f"Timestamp: {timestamp}\n")
    f.write(f"Status: {status}\n\n")

    if issues:
        f.write("Issues found:\n")
        for issue in issues:
            f.write(f"- {issue}\n")
    else:
        f.write("No issues found. All checks passed!\n")

print(f"Validation report generated:\n{report_path}")
print(f"Status: {status}")
if issues:
    print("Issues:")
    for issue in issues:
        print(f" - {issue}")

Validation report generated:
/content/drive/MyDrive/F1BigData/logs/merged_validation_2025-10-29_06-19-40.txt
Status: FAILED
Issues:
 - All LapTime values are missing.


##DuckDB

In [54]:
!pip install duckdb plotly --quiet

In [55]:
import duckdb
import plotly.express as px

lap_path = '/content/drive/MyDrive/F1BigData/cleaned_batches/final_clean_laps.csv'
tele_path = '/content/drive/MyDrive/F1BigData/cleaned_batches/telemetry_clean_fixed_2025-10-29_05-05-58.csv'

lap_df = pd.read_csv(lap_path)
tele_df = pd.read_csv(tele_path)

# register dataframes in DuckDB
con = duckdb.connect()
con.register("lap", lap_df)
con.register("tele", tele_df)

print("Data registered in DuckDB.")
print(f"Lap shape: {lap_df.shape}, Telemetry shape: {tele_df.shape}")

Data registered in DuckDB.
Lap shape: (1312, 31), Telemetry shape: (20014, 11)


In [64]:
# lap insights
con.sql("""
SELECT Driver,
       COUNT(*) AS laps
FROM lap
GROUP BY Driver
""").df()

,Driver,laps
0,PER,66
1,SAI,66
2,STR,66
3,ALO,66
4,ZHO,66
5,GAS,66
6,LEC,66
7,TSU,66
8,ALB,65
9,NOR,65


In [65]:
# telemetry insights
con.sql("""
SELECT
    AVG(Speed) AS avg_speed,
    MAX(RPM) AS max_rpm,
    AVG(Throttle) AS avg_throttle,
    AVG(CAST(Brake AS INTEGER)) AS avg_brake
FROM tele
""").df()

,avg_speed,max_rpm,avg_throttle,avg_brake
0,208.08589,12051.0,64.994554,0.166034


In [69]:
#find how each driver performed on tyre compounds, stint length
lap_insights = con.sql("""
SELECT
    Driver,
    Compound,
    COUNT(*) AS total_laps,
    AVG(LapTime) AS avg_lap_time,
    SUM(CASE WHEN IsPersonalBest THEN 1 ELSE 0 END) AS pb_laps
FROM lap
GROUP BY Driver, Compound
ORDER BY avg_lap_time
""").df()
lap_insights.head()
#insights: Which drivers consistently set PB laps, strategic distribution of tyre usage

,Driver,Compound,total_laps,avg_lap_time,pb_laps
0,RUS,SOFT,46,NaN,8.0
1,RUS,MEDIUM,20,NaN,2.0
2,PER,MEDIUM,27,NaN,6.0
3,PER,HARD,23,NaN,1.0
4,PER,SOFT,16,NaN,1.0


In [73]:
# car behavior and dynamics
tele_stats = con.sql("""
SELECT
    AVG(Speed) AS avg_speed,
    MAX(Speed) AS max_speed,
    MIN(Speed) AS min_speed,
    AVG(RPM) AS avg_rpm,
    AVG(Throttle) AS avg_throttle,
    AVG(CASE WHEN Brake THEN 1 ELSE 0 END)*100 AS brake_usage_pct,
    AVG(nGear) AS avg_gear
FROM tele
""").df()
tele_stats
# insights: avg and peak speeds, throttle smoothness, breaking frequency

,avg_speed,max_speed,min_speed,avg_rpm,avg_throttle,brake_usage_pct,avg_gear
0,208.08589,312.0,0.0,10177.153543,64.994554,16.603378,5.282152


In [75]:
speed_hist = con.sql("""
SELECT Speed FROM tele WHERE Speed IS NOT NULL
""").df()
px.histogram(speed_hist, x='Speed', nbins=40, title='Speed distribution (all drivers)', template='plotly_dark')

In [76]:
tele_corr = con.sql("""
SELECT Throttle, CAST(Brake AS INTEGER) AS Brake
FROM tele
WHERE Throttle IS NOT NULL
""").df()
px.scatter(tele_corr, x='Throttle', y='Brake', title='Throttle vs brake pattern', template='plotly_dark')

In [78]:
tele_segments = con.sql("""
SELECT
    ROUND(Distance/500)*500 AS distance_segment,
    AVG(Speed) AS avg_speed,
    AVG(RPM) AS avg_rpm
FROM tele
WHERE Distance IS NOT NULL
GROUP BY distance_segment
ORDER BY distance_segment
""").df()
px.line(tele_segments, x='distance_segment', y='avg_speed', title='Average speed per 500m segment', template='plotly_dark')

In [80]:
tele_trend = con.sql("""
SELECT
    CAST(SessionTime AS DOUBLE) AS time_sec,
    Speed,
    RPM
FROM tele
WHERE SessionTime IS NOT NULL
LIMIT 5000
""").df()

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=tele_trend['time_sec'], y=tele_trend['Speed'], mode='lines', name='Speed'))
fig.add_trace(go.Scatter(x=tele_trend['time_sec'], y=tele_trend['RPM'], mode='lines', name='RPM', yaxis='y2'))
fig.update_layout(
    title='Telemetry over session time',
    xaxis_title='SessionTime (s)',
    yaxis=dict(title='Speed (km/h)'),
    yaxis2=dict(title='RPM', overlaying='y', side='right'),
    template='plotly_dark')
fig.show()